# 01. 임베딩과 코사인 유사도 기초

목표: Chrome Embedding API를 쓰기 전에 임베딩 벡터와 유사도 비교가 무엇인지 직접 구현한다.

실행 방법:
1. Jupyter Notebook 또는 VS Code에서 이 파일을 연다.
2. 위에서 아래로 셀을 실행한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

주의: 여기의 임베딩은 학습용 장난감 구현이다. 실제 Chrome API는 브라우저 내장 모델이 `Float32Array` 벡터를 생성한다.

In [ ]:
import math
import re

# 단어를 소문자로 분리하는 아주 단순한 토크나이저다.
# 실제 임베딩 모델은 subword tokenizer와 신경망을 사용하지만,
# 여기서는 벡터의 의미를 이해하기 위해 직접 구현한다.
TOKEN_RE = re.compile(r"[a-zA-Z0-9가-힣]+")


def tokenize(text):
    return [token.lower() for token in TOKEN_RE.findall(text)]

## 1. 텍스트를 벡터로 바꾸기

가장 단순한 임베딩은 단어 등장 횟수를 벡터로 세는 것이다. 의미를 깊게 이해하지는 못하지만, 텍스트가 숫자 벡터가 된다는 핵심은 보여준다.

In [ ]:
texts = [
    "Chrome runs AI models on device",
    "Built-in AI APIs execute local models in the browser",
    "Server embeddings send text to a remote API",
    "Vector search compares semantic meaning",
]

# 전체 문서에서 나온 단어를 정렬해 고정된 차원의 vocabulary를 만든다.
# vocabulary 순서가 곧 벡터 차원의 의미가 된다.
vocabulary = sorted({token for text in texts for token in tokenize(text)})
vocabulary[:10], len(vocabulary)

In [ ]:
def count_embedding(text, vocabulary):
    """단어 등장 횟수 기반의 장난감 임베딩을 만든다."""
    tokens = tokenize(text)
    return [tokens.count(word) for word in vocabulary]


for text in texts[:2]:
    print(text)
    print(count_embedding(text, vocabulary))

## 2. 코사인 유사도

Embedding API 예제는 두 벡터의 유사도를 비교하기 위해 코사인 유사도 함수를 보여준다. 아래 구현은 같은 수학을 Python으로 옮긴 것이다.

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """두 벡터의 방향 유사도를 계산한다.

    분모가 0이면 방향이 정의되지 않으므로 0을 반환한다.
    실제 서비스 코드에서도 빈 입력이나 실패한 임베딩을 방어해야 한다.
    """
    if len(vec_a) != len(vec_b):
        raise ValueError("Vectors must have the same dimension")

    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))

    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

In [ ]:
query = "browser local AI model"
query_vec = count_embedding(query, vocabulary)

scores = []
for text in texts:
    text_vec = count_embedding(text, vocabulary)
    scores.append((cosine_similarity(query_vec, text_vec), text))

for score, text in sorted(scores, reverse=True):
    print(f"{score:.3f} | {text}")

## 3. 왜 실제 임베딩 모델이 필요한가

단어 카운트 방식은 `browser`와 `Chrome`, `local`과 `on-device` 같은 유사 표현을 연결하지 못한다. 실제 Embedding API는 내장 모델을 사용해 표면 단어가 달라도 의미가 가까운 문장을 더 가깝게 배치하려는 목적을 갖는다.

In [ ]:
pair_a = "Chrome runs AI models on device"
pair_b = "The browser executes local machine learning models"

vec_a = count_embedding(pair_a, vocabulary)
vec_b = count_embedding(pair_b, vocabulary)

print("toy similarity:", cosine_similarity(vec_a, vec_b))
print("학습 포인트: 실제 모델이라면 두 문장의 의미 유사도를 더 잘 잡을 수 있다.")

## 4. 같은 벡터 공간만 비교하기

원문에서 가장 중요한 주의사항은 같은 임베딩 공간에서 나온 벡터만 비교해야 한다는 점이다. 아래 예시는 차원이 다르면 비교 자체를 막는다.

In [ ]:
try:
    cosine_similarity([1, 2, 3], [1, 2])
except ValueError as error:
    print("비교 차단:", error)

print("실제 제품에서는 차원뿐 아니라 모델 버전과 taskType도 함께 확인해야 한다.")

## 정리

- 임베딩은 텍스트를 숫자 벡터로 바꾼 것이다.
- 코사인 유사도는 벡터 방향을 비교한다.
- 장난감 임베딩은 의미를 충분히 잡지 못하므로 실제 모델이 필요하다.
- 같은 벡터 공간에서 나온 벡터만 직접 비교해야 한다.